In [ ]:
%run NB00_setup

# NB01_dimensions
Builds the six dimensions: dim_date, dim_time, dim_anomaly_type (static), dim_vehicle and dim_route (SCD Type 1),
and dim_driver (SCD Type 2: drivers_v1 is loaded first, then drivers_v2 at the assumed change date).

In [ ]:
# ---- dim_date ----
dd = spark.sql(f"SELECT explode(sequence(to_date('{DATE_START}'), to_date('{DATE_END}'), interval 1 day)) AS full_date")
dim_date = dd.select(
    F.date_format("full_date", "yyyyMMdd").cast("int").alias("date_key"),
    "full_date",
    F.year("full_date").alias("year"),
    F.quarter("full_date").alias("quarter"),
    F.month("full_date").alias("month"),
    F.date_format("full_date", "MMMM").alias("month_name"),
    F.dayofmonth("full_date").alias("day"),
    F.dayofweek("full_date").alias("day_of_week"),
    F.date_format("full_date", "EEEE").alias("day_name"),
    F.weekofyear("full_date").alias("week_of_year"),
    F.dayofweek("full_date").isin(6, 7).alias("is_weekend"),   # Friday and Saturday (Spark: Sunday = 1)
)
merge_simple(dim_date, "gold_dim_date", "date_key")

# ---- dim_time (grain = one minute) ----
dim_time = (spark.range(0, 1440).select(F.col("id").cast("int").alias("minute_of_day"))
    .withColumn("hour", (F.col("minute_of_day") / 60).cast("int"))
    .withColumn("minute", (F.col("minute_of_day") % 60).cast("int"))
    .withColumn("time_key", F.col("hour") * 100 + F.col("minute"))
    .withColumn("time_label", F.format_string("%02d:%02d", F.col("hour"), F.col("minute")))
    .withColumn("day_part", F.when(F.col("hour") < 6, "night")
                             .when(F.col("hour") < 12, "morning")
                             .when(F.col("hour") < 18, "afternoon")
                             .otherwise("evening"))
    .select("time_key", "hour", "minute", "minute_of_day", "time_label", "day_part"))
merge_simple(dim_time, "gold_dim_time", "time_key")

# ---- dim_anomaly_type ----
rows = [
 (1, "cold_chain_breach", "Cold-chain breach", "cargo_temp_c outside 2-8 C for 3+ consecutive readings", "work plan"),
 (2, "excessive_idling",  "Excessive idling",  "speed < 2 km/h with rpm > 0 for 10+ minutes",            "work plan"),
 (3, "route_deviation",   "Route deviation",   "more than 2 km from corridor for 3+ readings",           "work plan"),
 (4, "engine_overheating","Engine overheating","engine temperature above plausible limit",               "added by Person 2"),
]
dim_at = spark.createDataFrame(rows, ["anomaly_type_key", "anomaly_type", "anomaly_name", "definition", "source"])
merge_simple(dim_at, "gold_dim_anomaly_type", "anomaly_type")

In [ ]:
# ---- dim_vehicle and dim_route (Type 1) ----
veh = spark.table("ref_vehicles").withColumn("is_refrigerated", F.col("vehicle_type") == "refrigerated_truck")
merge_type1_surrogate(veh, "gold_dim_vehicle", "vehicle_id", "vehicle_key")

rt = spark.table("ref_corridors")          # route = corridor
merge_type1_surrogate(rt, "gold_dim_route", "corridor_id", "route_key")

In [ ]:
# ---- dim_driver (Type 2) ----
DRV_COLS = ["driver_id", "full_name", "home_depot_id", "risk_level", "license_class", "hire_date"]

if not table_exists("gold_dim_driver"):
    spark.sql("""CREATE TABLE gold_dim_driver (
        driver_key BIGINT, driver_id STRING, full_name STRING, home_depot_id STRING,
        risk_level STRING, license_class STRING, hire_date DATE,
        effective_from TIMESTAMP, effective_to TIMESTAMP, is_current BOOLEAN) USING DELTA""")

def scd2_load_driver(src, change_ts=None):
    """Type 2 on home_depot_id, risk_level and license_class.
       A change is applied only when its date is newer than the current row, so loading v1 after v2
       does not move history backwards."""
    cur = (spark.table("gold_dim_driver").where("is_current = true")
           .select("driver_id",
                   F.col("home_depot_id").alias("c_depot"),
                   F.col("risk_level").alias("c_risk"),
                   F.col("license_class").alias("c_lic"),
                   F.col("effective_from").alias("c_eff")))
    s = src.select(*DRV_COLS, "effective_from").join(cur, "driver_id", "left")
    is_new  = F.col("c_eff").isNull()
    differs = ~(F.col("home_depot_id").eqNullSafe(F.col("c_depot")) &
                F.col("risk_level").eqNullSafe(F.col("c_risk")) &
                F.col("license_class").eqNullSafe(F.col("c_lic")))
    change_eff = F.lit(change_ts).cast("timestamp") if change_ts else F.col("effective_from")
    s = s.withColumn("eff", F.when(is_new, F.col("effective_from")).otherwise(change_eff))
    todo = s.where(is_new | (differs & (F.col("eff") > F.col("c_eff")))).cache()

    n_new = todo.where(is_new).count()
    n_chg = todo.where(~is_new).count()
    print(f"new drivers: {n_new} | changed drivers (new version): {n_chg}")
    if n_new + n_chg == 0:
        print("nothing to load (already up to date)")
        return

    base = max_key("gold_dim_driver", "driver_key")
    # rows to INSERT (merge_key = NULL never matches, so the MERGE inserts them)
    ins = (todo.withColumn("driver_key",
                (F.row_number().over(Window.orderBy("driver_id")) + F.lit(base)).cast("long"))
           .select("driver_key", *DRV_COLS, F.col("eff").alias("effective_from"),
                   F.lit(None).cast("string").alias("merge_key")))
    # rows that CLOSE the current version (merge_key = driver_id matches the current row)
    cls = (todo.where(~is_new)
           .select(F.lit(None).cast("long").alias("driver_key"), *DRV_COLS,
                   F.col("eff").alias("effective_from"),
                   F.col("driver_id").alias("merge_key")))
    staged = ins.unionByName(cls)

    (DeltaTable.forName(spark, "gold_dim_driver").alias("t")
        .merge(staged.alias("s"), "t.driver_id = s.merge_key AND t.is_current = true")
        .whenMatchedUpdate(set={"is_current": F.lit(False), "effective_to": F.col("s.effective_from")})
        .whenNotMatchedInsert(values={
            "driver_key": "s.driver_key", "driver_id": "s.driver_id", "full_name": "s.full_name",
            "home_depot_id": "s.home_depot_id", "risk_level": "s.risk_level",
            "license_class": "s.license_class", "hire_date": "s.hire_date",
            "effective_from": "s.effective_from",
            "effective_to": F.lit(HIGH_DATE).cast("timestamp"),
            "is_current": F.lit(True)})
        .execute())
    todo.unpersist()

# ---- load drivers v1 ----
scd2_load_driver(spark.table("ref_drivers_v1"))
d = spark.table("gold_dim_driver")
print("after v1 -> total rows:", d.count(), "| current rows:", d.where("is_current").count())

In [ ]:
# ---- load drivers v2 and show before/after (evidence for the SCD2 history) ----
v1 = spark.table("ref_drivers_v1")
v2 = spark.table("ref_drivers_v2")

print("effective_from values inside v2 file:")
v2.groupBy("effective_from").count().show(truncate=False)

chg = (v1.alias("a").join(v2.alias("b"), "driver_id")
       .where(~(F.col("a.home_depot_id").eqNullSafe(F.col("b.home_depot_id")) &
                F.col("a.risk_level").eqNullSafe(F.col("b.risk_level")) &
                F.col("a.license_class").eqNullSafe(F.col("b.license_class")))))
print("drivers with a real change in v2:", chg.count())
sample_ids = [r.driver_id for r in chg.select("driver_id").limit(5).collect()]

def show_drivers(title):
    print(title)
    (spark.table("gold_dim_driver").where(F.col("driver_id").isin(sample_ids))
          .orderBy("driver_id", "effective_from")
          .select("driver_key", "driver_id", "home_depot_id", "risk_level", "license_class",
                  "effective_from", "effective_to", "is_current")
          .show(truncate=False))

show_drivers("BEFORE v2:")
scd2_load_driver(v2, change_ts=V2_EFFECTIVE)
show_drivers("AFTER v2:")

bad = (spark.table("gold_dim_driver").where("is_current")
       .groupBy("driver_id").count().where("count != 1").count())
print("drivers without exactly one current row (expect 0):", bad)

In [ ]:
# ---- summary ----
for t in ["gold_dim_date", "gold_dim_time", "gold_dim_anomaly_type",
          "gold_dim_vehicle", "gold_dim_route", "gold_dim_driver"]:
    print(f"{t:28s} rows = {spark.table(t).count()}")

show_history("gold_dim_driver")